In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tqdm.auto import tqdm
import json
from itertools import product
from pathlib import Path
from itertools import product
import hashlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

In [ ]:
EXPERIMENT_NUMBER = 9  # variant: bins by distance to CH holdout (not Iceland)

# -------------------------------------------------
# Region code mapping (single source of truth)
# -------------------------------------------------
REGION_CODE_MAP = {
    "CH": "11_CH",
    "NOR": "08_NOR",
    "ISL": "06_ISL",
    "CEU": "11_CEU",
}

REGION_GROUPS = {
    "CEU": ["FR", "CH", "IT_AT"],
}

SOURCE_CODES = list(REGION_CODE_MAP.keys())
TARGET_CODES = list(REGION_CODE_MAP.keys())

MODEL = "AdapterLSTM"
BASE_DIR = Path(cfg.dataPath) / path_cache / f"{MODEL}_{EXPERIMENT_NUMBER}"

MODELS_DIR = BASE_DIR / "models"
CACHE_DIR = BASE_DIR / "cache"
RESULTS_DIR = BASE_DIR / "results"

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = [("ISL", "CH")]
PAIR_KEYS = ["ISL_to_CH"]

print(f"Pairs: (N = {len(PAIRS)})", PAIRS)
print("Pair keys:", PAIR_KEYS)

In [ ]:
vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

MONTHLY_COLS = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
    "ELEVATION_DIFFERENCE",
]
STATIC_COLS = ["aspect", "slope", "svf"]
feature_columns = MONTHLY_COLS + STATIC_COLS
feature_columns_ = feature_columns + ['POINT_BALANCE']

In [ ]:
BASE_GS_DIR = Path(cfg.dataPath) / path_cache / 'GS_results'
log_path_gs_results = {
    "ISL": BASE_GS_DIR / "lstm_param_search_progress_OOS_ISL_2026-02-11.csv",
    "NOR": BASE_GS_DIR / "lstm_param_search_progress_OOS_NOR_2026-02-09.csv",
    "CH": BASE_GS_DIR / "lstm_param_search_progress_CH_2026-02-18.csv",
}

for country, path in log_path_gs_results.items():
    if not path.exists():
        print(f"WARNING: {country} log file not found: {path}")

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

## Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
def resolve_region_codes(region, region_groups=None):
    if region_groups is None:
        region_groups = {}
    return region_groups.get(region, [region])


def filter_by_region(df, region, region_groups=None, source_col="SOURCE_CODE"):
    codes = resolve_region_codes(region, region_groups)
    return df[df[source_col].isin(codes)].copy()


def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode("utf-8")).hexdigest()[:10]

## Monthly datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_Europe.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_Europe.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")


## Experiment design:

In [ ]:
# load all stake dfs once
dfs = {
    rid: load_stakes_for_rgi_region(cfg, rid)
    for rid in tqdm(RGI_REGIONS.keys(), desc="Loading stake regions")
}

# recompute only pairs involving these symbolic/original region labels
# examples:
#   None          -> load all
#   {"CEU"}       -> recompute all pairs where src=="CEU" or tgt=="CEU"
#   {"ISL"}       -> recompute all pairs involving ISL
#   {"CEU","ISL"} -> recompute all pairs involving either CEU or ISL
#   "ALL"         -> recompute all
RECOMPUTE_REGIONS = None

all_pair_res = {}
all_pair_split_info = {}


def resolve_region(code, region_groups):
    """Return either the raw code or the grouped list of codes."""
    return region_groups.get(code, code)


def should_recompute_pair(src, tgt, recompute_regions=None):
    """
    Decide whether a pair should be recomputed.

    Parameters
    ----------
    src, tgt : str
        Pair labels as they appear in PAIRS, e.g. "CH", "ISL", "CEU".
    recompute_regions : set[str] | None | str
        - None  -> load all pairs
        - "ALL" -> recompute all pairs
        - set   -> recompute only pairs where src or tgt is in this set
    """
    if recompute_regions is None:
        return False
    if recompute_regions == "ALL":
        return True
    return (src in recompute_regions) or (tgt in recompute_regions)


for src, tgt in tqdm(PAIRS, desc="Preparing pairwise monthlies"):
    key = f"{src}_to_{tgt}"

    src_codes = resolve_region(src, REGION_GROUPS)
    tgt_codes = resolve_region(tgt, REGION_GROUPS)

    recompute_this = should_recompute_pair(
        src=src,
        tgt=tgt,
        recompute_regions=RECOMPUTE_REGIONS,
    )

    res_xreg, split_info = prepare_monthly_df_xreg_pairwise(
        cfg=cfg,
        dfs=dfs,
        paths=paths,
        vois_climate=vois_climate,
        vois_topographical=vois_topographical,
        source_code=src_codes,
        target_code=tgt_codes,
        run_flag=recompute_this,  # True = recompute, False = load
        region_name=f"XREG_{src}_TO_{tgt}",
        csv_subfolder=f"CrossRegional/{src}_to_{tgt}/csv",
    )

    all_pair_res[key] = res_xreg
    all_pair_split_info[key] = split_info

    df_train = res_xreg["df_train"]
    df_test = res_xreg["df_test"]

    status = "recomputed" if recompute_this else "loaded"

    print("\n" + "=" * 80)
    print(f"Pair: {key} [{status}]")
    print(f"Train glaciers ({src}):", len(split_info["train_glaciers"]))
    print(f"Test glaciers ({tgt}):", len(split_info["test_glaciers"]))
    print("Train rows:", len(df_train), "| Test rows:", len(df_test))

In [ ]:
df_sc_check = summarize_source_code_issues_in_all_pair_res(all_pair_res)
df_sc_check

In [ ]:
for key, res in all_pair_res.items():
    src_label, tgt_label = key.split("_to_")

    src_codes = resolve_region_codes(src_label, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt_label, REGION_GROUPS)

    df_test = res["df_test"]
    df_train = res["df_train"]

    # In pairwise setup, df_test should already be target-only and df_train source-only,
    # but we still filter explicitly for robustness.
    df_tgt = df_test[df_test["SOURCE_CODE"].isin(tgt_codes)].copy()
    df_src = df_train[df_train["SOURCE_CODE"].isin(src_codes)].copy()

    print("\n" + "=" * 80)
    print(f"Pair: {src_label} → {tgt_label}")

    print(f"{src_label} monthly rows:", len(df_src))
    print(f"{src_label} glaciers:", df_src["GLACIER"].nunique())
    if len(df_src) > 0:
        print(
            f"{src_label} year range:",
            df_src["YEAR"].min(),
            "-",
            df_src["YEAR"].max(),
        )
    else:
        print(f"No source rows found for {src_label}.")

    print(f"{tgt_label} monthly rows:", len(df_tgt))
    print(f"{tgt_label} glaciers:", df_tgt["GLACIER"].nunique())
    if len(df_tgt) > 0:
        print(
            f"{tgt_label} year range:",
            df_tgt["YEAR"].min(),
            "-",
            df_tgt["YEAR"].max(),
        )
    else:
        print(f"No target rows found for {tgt_label}.")

### Domain shift:

In [ ]:
shift = compute_domain_shift(
    df_src=res["df_train"],  # Iceland
    df_tgt=res["df_test"],  # Switzerland
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

# Summary distances
for k in [
        "D_mmd2_joint", "D_mmd2_climate", "D_mmd2_topo", "D_energy_joint",
        "D_energy_climate", "D_energy_topo"
]:
    print(f"{k:30s}: {shift[k]:.4f}")

# Per-variable ranking (most shifted first)
var_shifts = {
    k: v
    for k, v in shift.items() if k.startswith("D_mmd2_")
    and k not in ["D_mmd2_joint", "D_mmd2_climate", "D_mmd2_topo"]
}
print("\nMMD² per variable:")
for k, v in sorted(var_shifts.items(), key=lambda x: -x[1]):
    print(f"  {k:30s}: {v:.4f}")

var_energies = {
    k: v
    for k, v in shift.items() if k.startswith("D_energy_")
    and k not in ["D_energy_joint", "D_energy_climate", "D_energy_topo"]
}
print("\nEnergy distance per variable:")
for k, v in sorted(var_energies.items(), key=lambda x: -x[1]):
    print(f"  {k:30s}: {v:.4f}")

fig = plot_domain_shift(shift, MONTHLY_COLS, STATIC_COLS)
# fig.savefig("domain_shift.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
shift_with_pb = compute_domain_shift(
    df_src=res["df_train"],
    df_tgt=res["df_test"],
    monthly_cols=MONTHLY_COLS + ["POINT_BALANCE"],
    static_cols=STATIC_COLS,
)

mmd2_per_var = {
    k.replace("D_mmd2_", ""): v
    for k, v in shift_with_pb.items() if k.startswith("D_mmd2_")
    and k not in ["D_mmd2_joint", "D_mmd2_climate", "D_mmd2_topo"]
}

energy_per_var = {
    k.replace("D_energy_", ""): v
    for k, v in shift_with_pb.items() if k.startswith("D_energy_")
    and k not in ["D_energy_joint", "D_energy_climate", "D_energy_topo"]
}

wass_per_var = compute_wasserstein_per_var(
    res["df_train"],
    res["df_test"],
    cols=MONTHLY_COLS + STATIC_COLS + ["POINT_BALANCE"],
)

fig = plot_feature_kde_overlap(
    df_train=res_xreg['df_train'],
    df_test=res_xreg['df_test'],
    features=MONTHLY_COLS + STATIC_COLS + ['POINT_BALANCE'],
    label_dist_1="Iceland",
    label_dist_2="Switzerland",
)

for ax, feat in zip(fig.axes, MONTHLY_COLS + STATIC_COLS + ['POINT_BALANCE']):
    mmd2 = mmd2_per_var.get(feat)
    ed = energy_per_var.get(feat)
    wd = wass_per_var.get(feat)
    lines = []
    if mmd2 is not None: lines.append(f"MMD² = {mmd2:.3f}")
    if ed is not None: lines.append(f"E    = {ed:.3f}")
    if wd is not None: lines.append(f"W    = {wd:.3f}")
    if lines:
        ax.text(
            0.97,
            0.97,
            "\n".join(lines),
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8,
            fontweight="bold",
            color="black",
            bbox=dict(boxstyle="round,pad=0.2",
                      fc="white",
                      ec="none",
                      alpha=0.7),
        )

### Fixed glacier hold-out split (spatial generalization):
#### Select best holdout set:

In [ ]:
df_switzerland = res_xreg["df_test"]

best_split, best_score = None, np.inf

for s in range(100):
    split = split_pool_holdout(
        df_region=df_switzerland,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        holdout_frac=0.20,
        seed=s,
    )

    actual_frac = split["actual_holdout_frac"]
    if not (0.15 <= actual_frac <= 0.35):
        continue

    score = abs(split["mmd2_holdout_vs_region"]) + abs(
        split["mmd2_pool_vs_region"])
    if score < best_score:
        best_score = score
        best_split = split

# --- build dataframes ---
holdout_glaciers = best_split["holdout_glaciers"]
pool_glaciers = best_split["pool_glaciers"]

df_holdout = df_switzerland[df_switzerland["GLACIER"].isin(
    holdout_glaciers)].copy()
df_pool = df_switzerland[df_switzerland["GLACIER"].isin(pool_glaciers)].copy()

print(f"Holdout : {len(holdout_glaciers)} glaciers, {len(df_holdout)} rows")
print(f"Pool    : {len(pool_glaciers)} glaciers, {len(df_pool)} rows")
print(f"Holdout fraction (rows) : {len(df_holdout) / len(df_switzerland):.1%}")
print(f"MMD²(holdout, CH_all)   : {best_split['mmd2_holdout_vs_region']:.4f}")
print(f"MMD²(pool,    CH_all)   : {best_split['mmd2_pool_vs_region']:.4f}")
print(f"\nHoldout glaciers : {holdout_glaciers}")
print(f"Pool glaciers    : {pool_glaciers}")

pair_splits = {}
pair_splits["ISL_to_CH"] = {
    "source": src,
    "target": tgt,
    "df_src": df_src,
    "df_target": df_tgt,
    "df_pool": df_pool,
    "df_holdout": df_holdout,
    "pool_glaciers": pool_glaciers,
    "holdout_glaciers": holdout_glaciers,
}

#### Plot location:

In [ ]:
# --- 3) Example usage for any pair you already split ---
# pick a pair key, e.g. "ISL_to_CH"
key = "ISL_to_CH"
tgt = key.split("_to_")[1]

holdout_glaciers = pair_splits[key]["holdout_glaciers"]
pool_glaciers = pair_splits[key]["pool_glaciers"]

out = make_pool_holdout_map_for_target(cfg=cfg,
                                       target_code=tgt,
                                       holdout_glaciers=holdout_glaciers,
                                       pool_glaciers=pool_glaciers,
                                       TARGET_REGION_META=TARGET_REGION_META)

In [ ]:
fig = plot_feature_kde_overlap(df_pool,
                               df_holdout,
                               feature_columns_,
                               label_dist_1="Pool",
                               label_dist_2="Hold-out")

In [ ]:
data_glamos = load_stakes(cfg, "CH")

# Capitalize glacier names:
glacierCap = {}
for gl in data_glamos['GLACIER'].unique():
    if isinstance(gl, str):  # Ensure the glacier name is a string
        if gl.lower() == 'claridenu':
            glacierCap[gl] = 'Clariden_U'
        elif gl.lower() == 'claridenl':
            glacierCap[gl] = 'Clariden_L'
        else:
            glacierCap[gl] = gl.capitalize()
    else:
        print(f"Warning: Non-string glacier name encountered: {gl}")

fig = plot_heatmap(pool_glaciers,
                   data_glamos,
                   glacierCap,
                   period='annual',
                   cbar_label="Mean PMB [m w.e. $a^{-1}$]")

In [ ]:
figs = plot_tsne_overlap_src_vs_tgt_splits(
    res_src=res,  # your existing iceland result dict
    df_tgt_pool=df_pool,
    df_tgt_holdout=df_holdout,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    n_iter=1000,
    src_code='ISL',
    tgt_code='CH')

## Within-region modelling (safety check):

In [ ]:
TEST_GLACIERS_BY_CODE = {
    "CH": holdout_glaciers,
}

for k, v in TEST_GLACIERS_BY_CODE.items():
    print(k, len(v), v[:3])

df_CH = load_stakes(cfg, "CH")
df_CH["RGI_REGION"] = '11'
df_CH["SOURCE_CODE"] = 'CH'

dfs_grouped = {"CH": df_CH}

# Example:
res_all = prepare_monthly_dfs_for_all_regions(
    cfg=cfg,
    dfs=dfs_grouped,
    paths=paths,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    run_flag=False,
    test_glaciers_by_code=TEST_GLACIERS_BY_CODE,
    region_groups=None,
    only_codes=["CH"])

# Example usage:
# res_all is what you got from prepare_monthlies_for_all_regions(...)
figs = plot_overlap_for_all_results(
    results_dict=res_all,
    cfg=cfg,
    STATIC_COLS=STATIC_COLS,
    MONTHLY_COLS=MONTHLY_COLS,
    n_iter=1000,
)

In [ ]:
logs_cache_dir = BASE_DIR / "cache/LSTM_CH_baseline"
logs_cache_dir.mkdir(parents=True, exist_ok=True)

lstm_assets = build_or_load_lstm_all(
    cfg=cfg,
    res_all=res_all,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=logs_cache_dir,
    force_recompute=False,
    include_atomic=True,
    include_groups=False,
)

params_by_key = {
    'CH': {
        'Fm': 8,
        'Fs': 3,
        'hidden_size': 160,
        'num_layers': 2,
        'bidirectional': False,
        'dropout': 0.3,
        'static_layers': 2,
        'static_hidden': 32,
        'static_dropout': 0.1,
        'lr': 0.002,
        'weight_decay': 0.0001,
        'loss_name': 'neutral',
        'two_heads': False,
        'head_dropout': 0.1,
        'loss_spec': None
    }
}

models, infos = train_within_region_models_all(cfg=cfg,
                                               lstm_assets_by_key=lstm_assets,
                                               params_by_key=params_by_key,
                                               device=device,
                                               train_keys=["CH"],
                                               epochs=150,
                                               force_retrain=False,
                                               models_dir=MODELS_DIR)

In [ ]:
df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
    cfg=cfg,
    models_by_key=models,
    lstm_assets_by_key=lstm_assets,
    device=device,
    save_dir="figures/eval_within_region",
    ax_xlim=(-10, 10),
    ax_ylim=(-10, 10),
    grid_shape=(2, 2),
    grid_figsize=(18, 12))

display(df_metrics)

## Build static assets once (source + target holdout)

In [ ]:
def split_signature(glaciers):
    g = sorted(map(str, glaciers))
    return hashlib.md5((",".join(g)).encode("utf-8")).hexdigest()[:10]


print("ISL_to_CH", "holdout_sig:", split_signature(holdout_glaciers),
      "n_hold:", len(holdout_glaciers))

In [ ]:
static_assets_by_pair = {}
errors_static_assets = {}

pbar = tqdm(list(pair_splits.items()), desc="Building static TL assets")
for key, d in pbar:
    if "error" in d:
        errors_static_assets[key] = d["error"]
        continue

    src_label = d["source"]
    tgt_label = d["target"]

    src_codes = resolve_region_codes(src_label, REGION_GROUPS)
    tgt_codes = resolve_region_codes(tgt_label, REGION_GROUPS)

    print(src_codes, tgt_codes)

    pbar.set_postfix_str(f"{src_label}->{tgt_label}")

    res_xreg = all_pair_res[key]
    holdout_glaciers = d["holdout_glaciers"]

    sig = split_signature(holdout_glaciers)
    cache_dir_pair = CACHE_DIR / key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    def _build(force_recompute: bool):
        return build_static_tl_assets_source_and_holdout(
            cfg=cfg,
            res_xreg=res_xreg,
            target_codes=tgt_codes,
            source_codes=src_codes,
            target_label=tgt_label,
            source_label=src_label,
            holdout_glaciers=holdout_glaciers,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
            cache_dir=cache_dir_pair,
            force_recompute=force_recompute,
            show_progress=False,
        )

    try:
        force_recompute = False
        static_assets = _build(force_recompute=force_recompute)
        static_assets_by_pair[key] = static_assets

    except Exception as e:
        msg = repr(e)
        should_retry = ("Missing SOURCE_CODE for ds key" in msg) or isinstance(
            e, KeyError)

        if should_retry:
            try:
                static_assets = _build(force_recompute=True)
                static_assets_by_pair[key] = static_assets
            except Exception as e2:
                errors_static_assets[
                    key] = f"initial={msg} | recompute_failed={repr(e2)}"
        else:
            errors_static_assets[key] = msg

print(errors_static_assets)

## LSTM source baseline:

In [ ]:
BASE_GS_DIR = Path(cfg.dataPath) / path_cache / 'GS_results'
log_path_gs_results = {
    "ISL": BASE_GS_DIR / "lstm_param_search_progress_OOS_ISL_2026-02-11.csv",
    "NOR": BASE_GS_DIR / "lstm_param_search_progress_OOS_NOR_2026-02-09.csv",
    "CH": BASE_GS_DIR / "lstm_param_search_progress_CH_2026-02-18.csv",
}

for country, path in log_path_gs_results.items():
    if not path.exists():
        print(f"WARNING: {country} log file not found: {path}")

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)

params_by_key['11_CEU'] = params_by_key['11_CH']
params_by_key['01_USCA'] = default_params
params_by_key['01_CAW'] = default_params

baseline_models = {}
baseline_infos = {}

for src in tqdm(['ISL'], desc="Training/loading source baselines"):
    # pick any pair_key that uses this src, to get compatible tl_assets
    pair_key = next(k for k in static_assets_by_pair.keys()
                    if k.startswith(f"{src}_to_"))

    static_assets = static_assets_by_pair[pair_key]
    tl_assets_static = {"STATIC": static_assets}

    default_params = params_by_key[REGION_CODE_MAP[src]]
    print(default_params)

    model_src, model_path, info = train_or_load_source_baseline(
        cfg=cfg,
        tl_assets=tl_assets_static,
        default_params=default_params,
        device=device,
        source_code=src,
        models_dir=MODELS_DIR,
        prefix=f"lstm_{src}",  # e.g. lstm_CH
        key="BASELINE",
        train_flag=True,  # set False if you only want loading
        force_retrain=False,
        epochs=150,
        batch_size_train=64,
        batch_size_val=128,
        verbose=True,
        date=None,  # or None
    )

    baseline_models[src] = model_src
    baseline_infos[src] = {
        "path": model_path,
        "info": info,
        "pair_key_used": pair_key
    }

    print(f"{src}: baseline -> {model_path} (assets from {pair_key})")

## Define pools for experiments:

In [ ]:
def measurement_distances_to_source(
    df_pool: pd.DataFrame,
    df_source: pd.DataFrame,
    df_holdout: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    seed: int = 0,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute MMD² distance from each pool measurement (ID) to:
      1) source distribution (Iceland)
      2) holdout distribution (Swiss holdout)

    Uses shared scaling + bandwidths for consistency.

    Returns
    -------
    pd.DataFrame indexed by ID with:
        - mmd2_vs_iceland
        - mmd2_vs_holdout
    """

    def build_X(df):
        m = df.groupby(
            id_col)[monthly_cols].mean()  # annual mean climate per stake
        s = df.groupby(id_col)[static_cols].first()  # topo per stake
        return m.to_numpy(dtype=np.float64), s.to_numpy(
            dtype=np.float64), m.index.tolist()

    X_src_m, X_src_s, _ = build_X(df_source)
    X_pool_m, X_pool_s, pool_ids = build_X(df_pool)
    X_hold_m, X_hold_s, _ = build_X(df_holdout)

    scaler_m = StandardScaler().fit(np.vstack([X_pool_m, X_src_m, X_hold_m]))
    scaler_s = StandardScaler().fit(np.vstack([X_pool_s, X_src_s, X_hold_s]))

    X_src_z = np.hstack(
        [scaler_m.transform(X_src_m),
         scaler_s.transform(X_src_s)])
    X_pool_z = np.hstack(
        [scaler_m.transform(X_pool_m),
         scaler_s.transform(X_pool_s)])
    X_hold_z = np.hstack(
        [scaler_m.transform(X_hold_m),
         scaler_s.transform(X_hold_s)])

    # --- bandwidth: median heuristic on pooled subsample ---
    rng = np.random.default_rng(seed)
    Z = np.vstack([X_pool_z, X_src_z, X_hold_z])
    if Z.shape[0] > 2000:
        Z = Z[rng.choice(Z.shape[0], 2000, replace=False)]

    z2 = np.sum(Z * Z, axis=1, keepdims=True)
    D2 = np.maximum(z2 + z2.T - 2.0 * (Z @ Z.T), 0.0)
    np.fill_diagonal(D2, 0.0)
    median_d2 = np.median(D2[D2 > 0])
    sig = float(np.sqrt(0.5 * median_d2))
    bandwidths = [sig * 0.5, sig, sig * 2.0]  # MK-MMD

    def _rbf(A: np.ndarray, B: np.ndarray, sigma: float) -> np.ndarray:
        a2 = np.sum(A * A, axis=1, keepdims=True)
        b2 = np.sum(B * B, axis=1, keepdims=True).T
        return np.exp(-np.maximum(a2 + b2 - 2.0 * (A @ B.T), 0.0) /
                      (2.0 * sigma**2))

    # --- precompute E[k(y,y')] for a distribution ---
    def compute_Eyy(X):
        n = X.shape[0]
        out = []
        for sigma in bandwidths:
            K = _rbf(X, X, sigma)
            np.fill_diagonal(K, 0.0)
            out.append(float(K.sum() / (n * (n - 1))))
        return out

    Eyy_src = compute_Eyy(X_src_z)
    Eyy_hold = compute_Eyy(X_hold_z)

    # --- batched computation ---
    n_pool = X_pool_z.shape[0]
    n_batches = int(np.ceil(n_pool / batch_size))

    distances_src = np.empty(n_pool)
    distances_hold = np.empty(n_pool)

    for b in tqdm(range(n_batches), desc="MMD² distances"):
        start = b * batch_size
        end = min(start + batch_size, n_pool)
        batch = X_pool_z[start:end]

        dist_src = np.zeros(end - start)
        dist_hold = np.zeros(end - start)

        for sigma, Eyy_s, Eyy_h in zip(bandwidths, Eyy_src, Eyy_hold):
            Kxy_src = _rbf(batch, X_src_z, sigma)
            Kxy_hold = _rbf(batch, X_hold_z, sigma)

            # k(x,x) = 1 for RBF
            dist_src += 1.0 + Eyy_s - 2.0 * Kxy_src.mean(axis=1)
            dist_hold += 1.0 + Eyy_h - 2.0 * Kxy_hold.mean(axis=1)

        distances_src[start:end] = dist_src / len(bandwidths)
        distances_hold[start:end] = dist_hold / len(bandwidths)

    # --- return both distances ---
    return pd.DataFrame({
        "ID": pool_ids,
        "mmd2_vs_iceland": distances_src,
        "mmd2_vs_holdout": distances_hold,
    }).set_index("ID")


def build_experiment_grid(
    df_pool: pd.DataFrame,
    distances: pd.DataFrame,
    distance_col: str = "mmd2_vs_iceland",
    fractions: list[float] = [0.05, 0.10, 0.20, 0.50, 1.0],
    n_bins: int = 4,
    n_repeats: int = 5,
    id_col: str = "ID",
    seed: int = 0,
) -> tuple[list[dict], pd.Series]:

    # --- select distance used for binning ---
    if distance_col not in distances.columns:
        raise ValueError(f"{distance_col} not in distances columns")

    dist_series = distances[distance_col]

    # --- quantile bins ---
    bin_labels = [f"Q{i+1}" for i in range(n_bins)]
    binned = pd.qcut(dist_series, q=n_bins, labels=bin_labels)

    # --- optional glacier mapping ---
    id_to_glacier = (df_pool.groupby(id_col)["GLACIER"].first()
                     if "GLACIER" in df_pool.columns else None)

    rng = np.random.default_rng(seed)
    experiments = []
    total_jobs = n_bins * len(fractions) * n_repeats

    with tqdm(total=total_jobs, desc="Building experiment grid") as pbar:
        for bin_label in bin_labels:
            bin_ids = binned[binned == bin_label].index.tolist()
            total_meas = len(bin_ids)

            for frac in fractions:
                n_sample = max(1, int(np.round(frac * total_meas)))

                for rep in range(n_repeats):
                    sampled_ids = rng.choice(bin_ids,
                                             size=n_sample,
                                             replace=False).tolist()

                    # --- compute BOTH distance summaries ---
                    exp = {
                        "bin":
                        bin_label,
                        "fraction":
                        frac,
                        "repeat":
                        rep,
                        "ids":
                        sampled_ids,
                        "n_meas":
                        n_sample,

                        # main one used for binning
                        "mmd2_mean":
                        float(dist_series[sampled_ids].mean()),

                        # additional diagnostics
                        "mmd2_iceland_mean":
                        float(distances.loc[sampled_ids,
                                            "mmd2_vs_iceland"].mean()),
                        "mmd2_holdout_mean":
                        float(distances.loc[sampled_ids,
                                            "mmd2_vs_holdout"].mean()),
                    }

                    if id_to_glacier is not None:
                        exp["glaciers"] = (
                            id_to_glacier[sampled_ids].unique().tolist())

                    experiments.append(exp)

                    pbar.set_postfix(bin=bin_label,
                                     frac=f"{frac:.0%}",
                                     rep=rep)
                    pbar.update(1)

    # --- summary print ---
    print(
        f"\nBins ({n_bins} quantiles across {len(distances)} pool measurements, "
        f"based on '{distance_col}'):")

    for b in bin_labels:
        ids_in_bin = binned[binned == b].index

        glaciers_in_bin = (id_to_glacier[ids_in_bin].unique().tolist()
                           if id_to_glacier is not None else "n/a")

        print(f"  {b}: {len(ids_in_bin):4d} measurements | "
              f"MMD² [{dist_series[ids_in_bin].min():.3f}, "
              f"{dist_series[ids_in_bin].max():.3f}] | "
              f"glaciers: {glaciers_in_bin}")

    print(
        f"\nTotal experiments: {len(experiments)} "
        f"({n_bins} bins × {len(fractions)} fractions × {n_repeats} repeats)")

    return experiments, binned

In [ ]:
# ── wire up ───────────────────────────────────────────────────────────────────

df_iceland = res["df_train"]

meas_dist = measurement_distances_to_source(
    df_pool=df_pool,
    df_source=df_iceland,
    df_holdout=df_holdout,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

experiments, binned = build_experiment_grid(
    df_pool=df_pool,
    distances=meas_dist,
    fractions=np.arange(0.05, 1.05, 0.05),
    n_bins=6,
    n_repeats=5,
    distance_col="mmd2_vs_iceland",
)

In [ ]:
# Verify no ID leakage between pool and holdout
pool_ids = set(df_pool["ID"].unique())
holdout_ids = set(df_holdout["ID"].unique())
assert pool_ids.isdisjoint(holdout_ids), \
    f"Leakage: {len(pool_ids & holdout_ids)} IDs in both pool and holdout"

# Sanity check: all experiment IDs are from pool
all_exp_ids = set(mid for exp in experiments for mid in exp["ids"])
assert all_exp_ids.issubset(pool_ids), "Experiment contains non-pool IDs"

print(f"Pool IDs      : {len(pool_ids)}")
print(f"Holdout IDs   : {len(holdout_ids)}")
print(f"Experiment IDs: {len(all_exp_ids)} (subset of pool)")

In [ ]:
REBUILD_REGISTRY = True
N_JOBS = min(4, os.cpu_count() or 1)

registry_path = RESULTS_DIR / "experiment_registry_distance_bins.csv"

if registry_path.exists() and not REBUILD_REGISTRY:
    df_existing = pd.read_csv(registry_path)
    experiment_rows = df_existing.to_dict(orient="records")
    done_exp_keys = set(df_existing["exp_key"].astype(str))
    print(f"Loaded existing registry: {len(done_exp_keys)} experiments")
else:
    experiment_rows = []
    done_exp_keys = set()

for pair_key, d in tqdm(pair_splits.items(), desc="Pairs"):
    if "error" in d:
        continue

    src, tgt = d["source"], d["target"]
    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    sig = split_signature(d["holdout_glaciers"])
    cache_dir_pair = CACHE_DIR / pair_key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    pair_rows = []
    errs = []

    for exp in tqdm(experiments,
                    desc=f"{pair_key} — building assets",
                    leave=False):
        exp_key = (f"TL_{src}_to_{tgt}"
                   f"_bin{exp['bin']}"
                   f"_frac{int(exp['fraction']*100):03d}"
                   f"_rep{exp['repeat']:02d}")

        if exp_key in done_exp_keys and not REBUILD_REGISTRY:
            continue

        df_ft = df_pool[df_pool["ID"].isin(exp["ids"])].copy()
        if len(df_ft) == 0:
            errs.append({"exp_key": exp_key, "error": "empty df_ft"})
            continue

        try:
            _ = build_budget_assets_finetune_only(
                cfg=cfg,
                res_xreg=res_xreg,
                static_assets=static_assets,
                df_ft=df_ft,
                exp_key=exp_key,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
                cache_dir=cache_dir_pair,
                force_recompute=True,
                val_ratio=0.2,
                show_progress=False,
                seed=exp["repeat"],
            )

            pair_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key,
                "bin": exp["bin"],
                "fraction": exp["fraction"],
                "repeat": exp["repeat"],
                "n_meas": exp["n_meas"],
                "mmd2_mean":
                exp["mmd2_mean"],  # kept for back-compat (now = mmd2_holdout_mean, the binning distance)
                "mmd2_iceland_mean": exp["mmd2_iceland_mean"],  # ADD
                "mmd2_holdout_mean": exp["mmd2_holdout_mean"],  # ADD
                "glaciers_ft": ",".join(exp.get("glaciers", [])),
                "ids": ",".join(str(i) for i in exp["ids"]),
                "cache_dir": str(cache_dir_pair),
                "holdout_sig": sig,
            })
            done_exp_keys.add(exp_key)

        except Exception as e:
            errs.append({"exp_key": exp_key, "error": repr(e)})

    experiment_rows.extend(pair_rows)
    df_tmp = (pd.DataFrame(experiment_rows).drop_duplicates(subset=["exp_key"],
                                                            keep="last"))
    df_tmp.to_csv(registry_path, index=False)

    print(f"\n{pair_key}: registry_rows={len(df_tmp)} | "
          f"new_rows={len(pair_rows)} | errors={len(errs)}")
    if errs:
        print("  Errors:", errs[:5])

df_experiments = (pd.read_csv(registry_path).drop_duplicates(
    subset=["exp_key"], keep="last"))
print(f"\nTotal experiments in registry: {len(df_experiments)}")

### Compute E_ZERO:
E_ZERO = error of the baseline model evaluated on the fixed target holdout dataset (unseen glaciers), with no finetuning.

In [ ]:
RUN_E_ZERO = True

if RUN_E_ZERO:
    E0_rows = []
    MAKE_PLOTS = False

    for pair_key in tqdm(sorted(static_assets_by_pair.keys()),
                         desc="E_ZERO for all directed pairs"):
        src, tgt = pair_key.split("_to_")

        if src not in baseline_models:
            E0_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing baseline_model",
            })
            continue

        static_assets = static_assets_by_pair[pair_key]
        model_src = baseline_models[src]

        ax = None
        fig = None
        if MAKE_PLOTS:
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))

        try:
            metrics_zero, df_preds_zero, _, _ = evaluate_one_model_TL(
                cfg=cfg,
                model=model_src,
                device=device,
                tl_assets_for_key=static_assets,
                ax=ax,
                title=f"E_ZERO: {src} baseline on {tgt} holdout",
                batch_size=128,
                domain_vocab=static_assets.get("domain_vocab", None),
                show_plot=MAKE_PLOTS,
            )

            E_ZERO_a = float(metrics_zero["RMSE_annual"])
            E_ZERO_w = float(metrics_zero["RMSE_winter"])
            E_ZERO = float(np.mean([E_ZERO_a, E_ZERO_w]))

            row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "E_ZERO_a": E_ZERO_a,
                "E_ZERO_w": E_ZERO_w,
                "E_ZERO": E_ZERO,
            }

            for k in ["R2_annual", "R2_winter", "bias_annual", "bias_winter"]:
                if k in metrics_zero:
                    row[k] = metrics_zero[k]

            E0_rows.append(row)

            print(f"{pair_key}: E0_a={E_ZERO_a:.3f} | "
                  f"E0_w={E_ZERO_w:.3f} | E0={E_ZERO:.3f}")

            if MAKE_PLOTS:
                plt.show()

            del df_preds_zero

        except Exception as e:
            if MAKE_PLOTS and fig is not None:
                plt.close(fig)

            E0_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": repr(e),
            })

    df_E0 = pd.DataFrame(E0_rows).sort_values(["source", "target"])
    df_E0.to_csv(RESULTS_DIR / "df_E0_all_pairs.csv", index=False)

else:
    df_E0 = pd.read_csv(RESULTS_DIR / "df_E0_all_pairs.csv")

display(df_E0)

### E_TL:
#### Train adapter-only models for experiments:

In [ ]:
# load GS results:
with open(
        BASE_GS_DIR /
        "tuning_adapter/adapter_best_by_region_2026-02-24_15.json", "r") as f:
    best_by_region = json.load(f)

In [ ]:
def get_one_tl_asset_from_registry_row(
    *,
    cfg,
    row,
    pair_splits,
    static_assets_by_pair,
    all_pair_res,
    MONTHLY_COLS,
    STATIC_COLS,
    id_col: str = "ID",
):
    """
    Reconstruct one TL asset dict for one distance-bin experiment row.
    df_ft is reconstructed deterministically from the IDs stored in the
    registry row — no resampling needed.
    """
    pair_key = row["pair"]
    exp_key = row["exp_key"]
    seed = int(row["repeat"])

    d = pair_splits[pair_key]
    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    # reconstruct df_ft from pool using the sampled IDs stored at build time
    pool_glaciers = d["pool_glaciers"]
    df_pool_rows = d["df_target"][d["df_target"]["GLACIER"].isin(
        pool_glaciers)].copy()

    # IDs were stored as a comma-separated string in the registry
    sampled_ids = set(str(row["ids"]).split(",")) if isinstance(
        row["ids"], str) else set(row["ids"])
    df_ft = df_pool_rows[df_pool_rows[id_col].astype(str).isin(
        sampled_ids)].copy()

    if len(df_ft) == 0:
        raise ValueError(
            f"Empty df_ft for exp_key={exp_key} — IDs not found in pool")

    cache_dir_pair = Path(row["cache_dir"])

    assets_dict = build_budget_assets_finetune_only(
        cfg=cfg,
        res_xreg=res_xreg,
        static_assets=static_assets,
        df_ft=df_ft,
        exp_key=exp_key,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=cache_dir_pair,
        force_recompute=False,
        val_ratio=0.2,
        show_progress=False,
        seed=seed,
    )

    return assets_dict[exp_key]

In [ ]:
import os

# ── Delete stale output CSVs ──────────────────────────────────────────────────
results_eval_path = RESULTS_DIR / "df_etl_adapter_all.csv"
infos_eval_path = RESULTS_DIR / "df_tl_infos.csv"
errors_eval_path = RESULTS_DIR / "df_tl_errors.csv"

for path in [results_eval_path, infos_eval_path, errors_eval_path]:
    if path.exists():
        os.remove(path)
        print(f"Deleted: {path.name}")

# ── Reset all accumulators ────────────────────────────────────────────────────
results_rows = []
done_result_exp_keys = set()
infos_tl_rows = []
done_info_run_keys = set()
errors_tl_rows = []
n_since_save = 0
print("Accumulators reset.")

# ── Patch ids from in-memory experiments list into df_experiments ─────────────
id_lookup = {
    f"TL_ISL_to_CH_bin{exp['bin']}_frac{int(exp['fraction']*100):03d}_rep{exp['repeat']:02d}":
    ",".join(str(i) for i in exp["ids"])
    for exp in experiments
}

df_experiments = pd.read_csv(RESULTS_DIR /
                             "experiment_registry_distance_bins.csv")
df_experiments["ids"] = df_experiments["exp_key"].map(id_lookup)

assert df_experiments["ids"].notna().all(), \
    f"Patch failed: {df_experiments['ids'].isna().sum()} rows unmapped"

df_experiments.to_csv(RESULTS_DIR / "experiment_registry_distance_bins.csv",
                      index=False)
print(f"df_experiments ready: {len(df_experiments)} rows, ids column patched.")

In [ ]:
RUN_EVAL = True
RELOAD_EXISTING_EVAL = False  # False = fresh run, True = resume
SAVE_EVERY = 1

E0_lookup = (df_E0.set_index("pair")[["E_ZERO_a",
                                      "E_ZERO_w"]].to_dict(orient="index"))

# ── Optionally resume from existing CSVs ─────────────────────────────────────
if RELOAD_EXISTING_EVAL and results_eval_path.exists():
    df_results_existing = pd.read_csv(results_eval_path)
    results_rows = df_results_existing.to_dict(orient="records")
    done_result_exp_keys = set(df_results_existing["exp_key"].astype(str))
    print(f"Resuming: {len(done_result_exp_keys)} experiments already done")
else:
    results_rows = []
    done_result_exp_keys = set()

if RELOAD_EXISTING_EVAL and infos_eval_path.exists():
    df_infos_existing = pd.read_csv(infos_eval_path)
    infos_tl_rows = df_infos_existing.to_dict(orient="records")
    done_info_run_keys = set(df_infos_existing["run_key"].astype(str))
else:
    infos_tl_rows = []
    done_info_run_keys = set()

if RELOAD_EXISTING_EVAL and errors_eval_path.exists():
    errors_tl_rows = pd.read_csv(errors_eval_path).to_dict(orient="records")
else:
    errors_tl_rows = []

n_since_save = 0

# ── Main loop ─────────────────────────────────────────────────────────────────
for row in tqdm(
        df_experiments.sort_values(["pair", "bin", "fraction",
                                    "repeat"]).to_dict(orient="records"),
        desc="Train + evaluate TL experiments",
):
    pair_key = row["pair"]
    src = row["source"]
    tgt = row["target"]
    exp_key = row["exp_key"]
    run_key = f"{exp_key}__adapter"

    if exp_key in done_result_exp_keys:
        continue

    try:
        # parse ids (comma-separated string from CSV)
        raw_ids = row.get("ids", None)
        if raw_ids is None or (isinstance(raw_ids, float)
                               and np.isnan(raw_ids)):
            raise KeyError(f"ids missing for {exp_key}")
        sampled_ids = [i.strip() for i in str(raw_ids).split(",") if i.strip()]

        assets = get_one_tl_asset_from_registry_row(
            cfg=cfg,
            row={
                **row, "ids": sampled_ids
            },
            pair_splits=pair_splits,
            static_assets_by_pair=static_assets_by_pair,
            all_pair_res=all_pair_res,
            MONTHLY_COLS=MONTHLY_COLS,
            STATIC_COLS=STATIC_COLS,
        )

        d = pair_splits[pair_key]
        sig = split_signature(d["holdout_glaciers"])
        model_dir_pair = MODELS_DIR / pair_key / f"holdout_{sig}"
        model_dir_pair.mkdir(parents=True, exist_ok=True)

        model, out_path, info = finetune_or_load_one_TL(
            cfg=cfg,
            exp_key=exp_key,
            tl_assets_for_key=assets,
            best_params=params_by_key[REGION_CODE_MAP[src]],
            device=device,
            pretrained_ckpt_path=baseline_infos[src]["path"],
            models_dir=model_dir_pair,
            prefix=f"lstm_TL_{src}_to_{tgt}",
            strategy="adapter",
            force_retrain=False,
            verbose=False,
            best_by_region=best_by_region,
            date=None,
            load_latest=True,
            skip_if_missing=False,
            prefer_tuned_ckpt=False,
        )

        if run_key not in done_info_run_keys:
            infos_tl_rows.append({
                "pair":
                pair_key,
                "source":
                src,
                "target":
                tgt,
                "exp_key":
                exp_key,
                "run_key":
                run_key,
                "out_path":
                str(out_path),
                **({
                    k: v
                    for k, v in info.items()
                } if isinstance(info, dict) else {
                       "info": info
                   }),
            })
            done_info_run_keys.add(run_key)

        if not RUN_EVAL:
            del assets, model, out_path, info
            continue

        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=model,
            device=device,
            tl_assets_for_key=assets,
            ax=None,
            title=None,
            batch_size=128,
            domain_vocab=assets.get("domain_vocab", None),
            show_plot=False,
        )

        E0_a = E0_lookup.get(pair_key, {}).get("E_ZERO_a", float("nan"))
        E0_w = E0_lookup.get(pair_key, {}).get("E_ZERO_w", float("nan"))

        results_rows.append({
            **row,
            **metrics,
            "run_key":
            run_key,
            "pair":
            pair_key,
            "source":
            src,
            "target":
            tgt,
            "E_ZERO_RMSE_annual":
            E0_a,
            "Delta_vs_ZERO_annual":
            metrics["RMSE_annual"] - E0_a,
            "E_ZERO_RMSE_winter":
            E0_w,
            "Delta_vs_ZERO_winter":
            metrics["RMSE_winter"] - E0_w,
        })
        done_result_exp_keys.add(exp_key)
        del assets, model, out_path, info, df_preds

    except Exception as e:
        errors_tl_rows.append({**row, "run_key": run_key, "error": repr(e)})

    # incremental save
    n_since_save += 1
    if n_since_save >= SAVE_EVERY:
        if infos_tl_rows:
            pd.DataFrame(infos_tl_rows).drop_duplicates(
                "run_key", keep="last").to_csv(infos_eval_path, index=False)
        if errors_tl_rows:
            pd.DataFrame(errors_tl_rows).to_csv(errors_eval_path, index=False)
        if results_rows:
            pd.DataFrame(results_rows).drop_duplicates(
                "exp_key", keep="last").to_csv(results_eval_path, index=False)
        n_since_save = 0

# ── Final save ────────────────────────────────────────────────────────────────
df_tl_infos = pd.DataFrame(infos_tl_rows).drop_duplicates(
    "run_key", keep="last") if infos_tl_rows else pd.DataFrame()
df_tl_errors = pd.DataFrame(
    errors_tl_rows) if errors_tl_rows else pd.DataFrame()

if results_rows:
    df_etl_adapter_all = (pd.DataFrame(results_rows).drop_duplicates(
        "exp_key", keep="last").set_index("exp_key").sort_index())
    df_etl_adapter_all.to_csv(results_eval_path)
else:
    df_etl_adapter_all = pd.DataFrame()

df_tl_infos.to_csv(infos_eval_path, index=False)
df_tl_errors.to_csv(errors_eval_path, index=False)

print(
    f"\nDone: {len(df_etl_adapter_all)} results | {len(df_tl_errors)} errors | {len(df_tl_infos)} infos"
)
if not df_tl_errors.empty:
    print(df_tl_errors[["exp_key",
                        "error"]].drop_duplicates("error").to_string())
display(df_etl_adapter_all.head())

### Compute E_FULL: 
Adapter fine-tuned on all ISL pool data (everything that is not in the fixed holdout glaciers), evaluated on the same fixed holdout (ds_test).

In [ ]:
from pathlib import Path


def get_one_fullpool_asset(
    *,
    cfg,
    pair_key,
    pair_splits,
    static_assets_by_pair,
    all_pair_res,
    MONTHLY_COLS,
    STATIC_COLS,
):
    d = pair_splits[pair_key]
    src = d["source"]
    tgt = d["target"]

    static_assets = static_assets_by_pair[pair_key]
    res_xreg = all_pair_res[pair_key]

    pool_glaciers = d["pool_glaciers"]
    df_pool_rows = d["df_target"][d["df_target"]["GLACIER"].isin(
        pool_glaciers)].copy()

    if len(df_pool_rows) == 0:
        raise ValueError("empty df_pool_rows")

    sig = split_signature(d["holdout_glaciers"])
    cache_dir_pair = Path(CACHE_DIR) / pair_key / f"holdout_{sig}"
    cache_dir_pair.mkdir(parents=True, exist_ok=True)

    exp_key_full = f"TL_{src}_to_{tgt}_FULLPOOL"

    assets_dict = build_budget_assets_finetune_only(
        cfg=cfg,
        res_xreg=res_xreg,
        static_assets=static_assets,
        df_ft=df_pool_rows,
        exp_key=exp_key_full,
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=cache_dir_pair,
        force_recompute=False,
        val_ratio=0.2,
        show_progress=False,
        seed=cfg.seed,
    )

    return exp_key_full, assets_dict[exp_key_full], df_pool_rows

In [ ]:
RUN_E_FULL = True

if RUN_E_FULL:
    EFULL_rows = []
    infos_full_rows = []
    errors_full_rows = []

    for pair_key in tqdm(sorted(pair_splits.keys()),
                         desc="FULLPOOL train + eval"):
        d = pair_splits[pair_key]

        if "error" in d:
            errors_full_rows.append({
                "pair": pair_key,
                "source": d.get("source"),
                "target": d.get("target"),
                "error": d["error"],
            })
            continue

        src = d["source"]
        tgt = d["target"]

        if pair_key not in static_assets_by_pair:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing static_assets_by_pair",
            })
            continue

        if src not in baseline_infos:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": "missing baseline_infos",
            })
            continue

        try:
            # 1) build/load one FULLPOOL asset
            exp_key_full, assets_full, df_pool_rows = get_one_fullpool_asset(
                cfg=cfg,
                pair_key=pair_key,
                pair_splits=pair_splits,
                static_assets_by_pair=static_assets_by_pair,
                all_pair_res=all_pair_res,
                MONTHLY_COLS=MONTHLY_COLS,
                STATIC_COLS=STATIC_COLS,
            )

            # 2) model dir
            sig = split_signature(d["holdout_glaciers"])
            model_dir_pair = MODELS_DIR / pair_key / f"holdout_{sig}"
            model_dir_pair.mkdir(parents=True, exist_ok=True)

            # 3) source-specific baseline ckpt + params
            pretrained_ckpt_path = baseline_infos[src]["path"]
            best_params_src = params_by_key[REGION_CODE_MAP[src]]

            # 4) finetune/load one FULLPOOL model
            model_full, out_path_full, info_full = finetune_or_load_one_TL(
                cfg=cfg,
                exp_key=exp_key_full,
                tl_assets_for_key=assets_full,
                best_params=best_params_src,
                device=device,
                pretrained_ckpt_path=pretrained_ckpt_path,
                models_dir=model_dir_pair,
                prefix=f"lstm_TL_{src}_to_{tgt}",
                strategy="adapter",
                force_retrain=False,
                verbose=False,
                best_by_region=best_by_region,
                date=None,
                load_latest=True,
                skip_if_missing=False,
                prefer_tuned_ckpt=False,
            )

            infos_full_rows.append({
                "pair":
                pair_key,
                "source":
                src,
                "target":
                tgt,
                "exp_key":
                exp_key_full,
                "run_key":
                f"{exp_key_full}__adapter",
                "out_path":
                str(out_path_full),
                **({
                    k: v
                    for k, v in info_full.items()
                } if isinstance(info_full, dict) else {
                       "info": info_full
                   }),
            })

            # 5) evaluate immediately
            metrics_full, df_preds_full, _, _ = evaluate_one_model_TL(
                cfg=cfg,
                model=model_full,
                device=device,
                tl_assets_for_key=assets_full,
                ax=None,
                title=None,
                batch_size=128,
                domain_vocab=assets_full.get("domain_vocab", None),
                show_plot=False,
            )

            E_FULL_a = float(metrics_full["RMSE_annual"])
            E_FULL_w = float(metrics_full["RMSE_winter"])
            E_FULL = float(np.mean([E_FULL_a, E_FULL_w]))

            row = {
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "exp_key": exp_key_full,
                "run_key": f"{exp_key_full}__adapter",
                "n_rows_ft": int(len(df_pool_rows)),
                "n_glaciers_ft": int(df_pool_rows["GLACIER"].nunique()),
                "n_years_ft": int(df_pool_rows["YEAR"].nunique()),
                "E_FULL_a": E_FULL_a,
                "E_FULL_w": E_FULL_w,
                "E_FULL": E_FULL,
            }

            for k in ["R2_annual", "R2_winter", "bias_annual", "bias_winter"]:
                if k in metrics_full:
                    row[k] = metrics_full[k]

            EFULL_rows.append(row)

            print(f"{pair_key}: E_FULL_a={E_FULL_a:.3f} | "
                  f"E_FULL_w={E_FULL_w:.3f} | E_FULL={E_FULL:.3f}")

            del assets_full, model_full, out_path_full, info_full, df_preds_full

        except Exception as e:
            errors_full_rows.append({
                "pair": pair_key,
                "source": src,
                "target": tgt,
                "error": repr(e),
            })

    df_EFULL = pd.DataFrame(EFULL_rows).sort_values(["source", "target"])
    df_EFULL.to_csv(RESULTS_DIR / "df_EFULL_all_pairs.csv", index=False)

    df_EFULL_infos = pd.DataFrame(infos_full_rows)
    df_EFULL_errors = pd.DataFrame(errors_full_rows)

else:
    df_EFULL = pd.read_csv(RESULTS_DIR / "df_EFULL_all_pairs.csv")

display(df_EFULL)

## Evalute experiment:

In [ ]:
results_eval_path = RESULTS_DIR / "df_etl_adapter_all.csv"

df_etl_adapter_all = (pd.read_csv(results_eval_path).drop_duplicates(
    subset=["exp_key"], keep="last").set_index("exp_key").sort_index())

print(f"Loaded {len(df_etl_adapter_all)} experiments")
print(f"Columns: {df_etl_adapter_all.columns.tolist()}")
display(df_etl_adapter_all.head())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats

# ── 0. Load results ───────────────────────────────────────────────────────────

df = df_etl_adapter_all.copy().reset_index()

df["bin"] = df["bin"].astype(str)
df["fraction"] = df["fraction"].astype(float)
df["repeat"] = df["repeat"].astype(int)

# Derive BIN_ORDER dynamically from actual data
n_bins = df["bin"].nunique()
BIN_ORDER = [f"Q{i+1}" for i in range(n_bins)]

BIN_LABELS = {
    BIN_ORDER[0]:  f"Q1 — closest to Iceland",
    BIN_ORDER[-1]: f"Q{n_bins} — furthest from Iceland",
    **{b: b for b in BIN_ORDER[1:-1]},  # middle bins just get their label
}

# Generate enough colors for n_bins (interpolate from a palette)
import matplotlib.cm as mcm
_cmap = mcm.get_cmap("RdYlBu_r", n_bins)
BIN_COLORS = {b: "#{:02x}{:02x}{:02x}".format(
    *[int(c * 255) for c in _cmap(i)[:3]]
) for i, b in enumerate(BIN_ORDER)}

FRAC_COLORS = {
    0.05: "#B5D4F4",
    0.10: "#85B7EB",
    0.15: "#378ADD",
    0.20: "#185FA5",
    0.30: "#0C447C",
    0.50: "#042C53",
    0.75: "#26215C",
    1.00: "#3C3489"
}

# ── 1. Aggregate: mean + std over repeats ─────────────────────────────────────

agg = (
    df.groupby(["bin", "fraction"]).agg(
        RMSE_annual_mean=("RMSE_annual", "mean"),
        RMSE_annual_std=("RMSE_annual", "std"),
        RMSE_winter_mean=("RMSE_winter", "mean"),
        RMSE_winter_std=("RMSE_winter", "std"),
        R2_annual_mean=("R2_annual", "mean"),
        R2_annual_std=("R2_annual", "std"),
        mmd2_iceland_mean=("mmd2_iceland_mean", "mean"),  # was mmd2_mean
        mmd2_holdout_mean=("mmd2_holdout_mean", "mean"),  # NEW
        n_meas_mean=("n_meas", "mean"),
    ).reset_index())
agg["bin"] = pd.Categorical(agg["bin"], categories=BIN_ORDER, ordered=True)
agg = agg.sort_values(["bin", "fraction"])

fractions = sorted(df["fraction"].unique())
pct_labels = [f"{int(f*100)}%" for f in fractions]


def plot_quantity_effect(metric="RMSE_annual", ylabel="RMSE annual (m w.e.)"):
    """Left plot: RMSE vs fraction, one line per bin."""
    fig, ax = plt.subplots(figsize=(7, 5))

    for b in BIN_ORDER:
        sub = agg[agg["bin"] == b].sort_values("fraction")
        mu = sub[f"{metric}_mean"].values
        sd = sub[f"{metric}_std"].values
        xs = sub["fraction"].values
        c = BIN_COLORS[b]

        ax.plot(xs,
                mu,
                color=c,
                linewidth=2,
                marker="o",
                markersize=5,
                label=BIN_LABELS[b],
                zorder=3)
        ax.fill_between(xs, mu - sd, mu + sd, color=c, alpha=0.12, zorder=2)

    # reference lines
    ax.axhline(E0_a if "annual" in metric else E0_w,
               color="#888780",
               linewidth=1.2,
               linestyle="--",
               label="Zero-shot (no FT)")
    ax.axhline(EF_a if "annual" in metric else EF_w,
               color="#444441",
               linewidth=1.2,
               linestyle=":",
               label="Full pool FT")

    ax.set_xticks(fractions)
    ax.set_xticklabels(pct_labels, fontsize=10)
    ax.set_xlabel("Fraction of distance bin sampled", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.6)
    ax.set_axisbelow(True)
    fig.tight_layout()
    return fig


def plot_quality_effect(metric="RMSE_annual", ylabel="RMSE annual (m w.e.)"):
    """Right plot: RMSE vs bin at fixed fractions."""
    fig, ax = plt.subplots(figsize=(7, 5))

    show_fracs = [f for f in [0.05, 0.20, 1.0] if f in fractions]
    n = len(show_fracs)
    width = 0.2
    x = np.arange(len(BIN_ORDER))

    for i, frac in enumerate(show_fracs):
        # FIX: use the already-aggregated `agg` df, not raw `df`
        sub = (
            agg[agg["fraction"] == frac]
            .assign(bin=lambda d: d["bin"].astype(str))
            .set_index("bin")
            .reindex(BIN_ORDER)
        )
        mu = sub[f"{metric}_mean"].values
        sd = sub[f"{metric}_std"].values
        offset = (i - n / 2 + 0.5) * width
        c = list(FRAC_COLORS.values())[i * 2]

        bars = ax.bar(x + offset, mu, width=width * 0.85, color=c,
                      label=f"{int(frac*100)}%", zorder=3)
        ax.errorbar(x + offset, mu, yerr=sd, fmt="none", color="#444441",
                    linewidth=1, capsize=3, zorder=4)

    ax.axhline(E0_a if "annual" in metric else E0_w,
               color="#888780", linewidth=1.2, linestyle="--", label="Zero-shot")
    ax.axhline(EF_a if "annual" in metric else EF_w,
               color="#444441", linewidth=1.2, linestyle=":", label="Full pool FT")

    ax.set_xticks(x)
    ax.set_xticklabels([BIN_LABELS[b] for b in BIN_ORDER],
                       fontsize=9, rotation=15, ha="right")
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(frameon=False, fontsize=9, title="Fraction sampled")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.6)
    ax.set_axisbelow(True)
    fig.tight_layout()
    return fig


def plot_both_distances_vs_rmse(metric="RMSE_annual",
                                ylabel="RMSE annual (m w.e.)"):
    """Side-by-side scatter: MMD² vs Iceland and MMD² vs holdout."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, dist_col, xlabel in zip(
            axes,
        ["mmd2_iceland_mean", "mmd2_holdout_mean"],
        [
            "MMD²(fine-tuning sample, Iceland)",
            "MMD²(fine-tuning sample, CH holdout)"
        ],
    ):
        for b in BIN_ORDER:
            sub = df[df["bin"] == b]
            ax.scatter(
                sub[dist_col],
                sub[metric],
                c=BIN_COLORS[b],
                s=sub["n_meas"] / sub["n_meas"].max() * 120 + 10,
                alpha=0.55,
                linewidths=0,
                label=BIN_LABELS[b],
                zorder=3,
            )

        valid = df[[dist_col, metric]].dropna()
        r, p = stats.spearmanr(valid[dist_col], valid[metric])
        ax.text(0.03,
                0.96,
                f"Spearman r = {r:.2f}  (p = {p:.3f})",
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                color="#444441")

        ax.axhline(E0_a if "annual" in metric else E0_w,
                   color="#888780",
                   linewidth=1,
                   linestyle="--",
                   label="Zero-shot")
        ax.axhline(EF_a if "annual" in metric else EF_w,
                   color="#444441",
                   linewidth=1,
                   linestyle=":",
                   label="Full pool FT")

        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(frameon=False, fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(color="#e0e0e0", linewidth=0.6)
        ax.set_axisbelow(True)

    fig.tight_layout()
    return fig


def plot_delta_heatmap(metric="Delta_vs_ZERO_annual",
                       title="RMSE improvement over zero-shot — annual"):
    """Heatmap: bins × fractions, colour = mean delta vs zero-shot."""
    pivot = (df.groupby(
        ["bin",
         "fraction"])[metric].mean().unstack("fraction").reindex(BIN_ORDER))
    pivot.columns = [f"{int(f*100)}%" for f in pivot.columns]

    fig, ax = plt.subplots(figsize=(9, 4))
    vmax = max(abs(pivot.values[np.isfinite(pivot.values)]))
    im = ax.imshow(pivot.values,
                   aspect="auto",
                   cmap="RdBu_r",
                   vmin=-vmax,
                   vmax=vmax)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=10)
    ax.set_yticks(range(len(BIN_ORDER)))
    ax.set_yticklabels([BIN_LABELS[b] for b in BIN_ORDER], fontsize=10)
    ax.set_xlabel("Fraction of bin sampled", fontsize=11)
    ax.set_ylabel("Distance bin", fontsize=11)

    # annotate cells
    for i in range(len(BIN_ORDER)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if np.isfinite(val):
                ax.text(j,
                        i,
                        f"{val:+.3f}",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="white" if abs(val) > vmax * 0.6 else "#2C2C2A")

    plt.colorbar(im, ax=ax, label="ΔRMSE vs zero-shot (negative = better)")
    ax.set_title(title, fontsize=12, fontweight="normal", pad=10)
    fig.tight_layout()
    return fig

In [ ]:
# ── Generate all plots ────────────────────────────────────────────────────────

fig_qty_a = plot_quantity_effect("RMSE_annual", "RMSE annual (m w.e.)")
fig_qty_w = plot_quantity_effect("RMSE_winter", "RMSE winter (m w.e.)")
fig_qual_a = plot_quality_effect("RMSE_annual", "RMSE annual (m w.e.)")
fig_qual_w = plot_quality_effect("RMSE_winter", "RMSE winter (m w.e.)")
fig_scat_a = plot_both_distances_vs_rmse("RMSE_annual",
                                         "RMSE annual (m w.e.)")  # replaced
fig_scat_w = plot_both_distances_vs_rmse("RMSE_winter",
                                         "RMSE winter (m w.e.)")  # new
fig_heat_a = plot_delta_heatmap("Delta_vs_ZERO_annual",
                                "ΔRMSE vs zero-shot — annual")
fig_heat_w = plot_delta_heatmap("Delta_vs_ZERO_winter",
                                "ΔRMSE vs zero-shot — winter")

for name, fig in {
        "quantity_annual": fig_qty_a,
        "quantity_winter": fig_qty_w,
        "quality_annual": fig_qual_a,
        "quality_winter": fig_qual_w,
        "both_distances_annual": fig_scat_a,
        "both_distances_winter": fig_scat_w,
        "heatmap_annual": fig_heat_a,
        "heatmap_winter": fig_heat_w,
}.items():
    fig.savefig(RESULTS_DIR / f"distbin_{name}.png",
                dpi=150,
                bbox_inches="tight")

plt.show()

In [ ]:
from scipy.stats import pearsonr
import statsmodels.formula.api as smf

df_model = df[[
    "RMSE_annual", "mmd2_iceland_mean", "mmd2_holdout_mean", "n_meas"
]].dropna()

# Partial correlation: does mmd2_iceland_mean predict RMSE
# after controlling for mmd2_holdout_mean and n_meas?
result = smf.ols(
    "RMSE_annual ~ mmd2_iceland_mean + mmd2_holdout_mean + np.log1p(n_meas)",
    data=df_model).fit()

print(result.summary())

In [ ]:
print(df[["mmd2_iceland_mean", "mmd2_holdout_mean"]].corr())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
import pandas as pd
import numpy as np

df_model = df[[
    "RMSE_annual", "mmd2_iceland_mean", "mmd2_holdout_mean", "n_meas"
]].dropna()
df_model["log_n_meas"] = np.log1p(df_model["n_meas"])

X = df_model[["mmd2_iceland_mean", "mmd2_holdout_mean", "log_n_meas"]]
vif = pd.DataFrame({
    "feature":
    X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif)

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

df_model = df[[
    "RMSE_annual", "mmd2_iceland_mean", "mmd2_holdout_mean", "n_meas"
]].dropna()
df_model["log_n_meas"] = np.log1p(df_model["n_meas"])

# Model A: Iceland distance only
res_a = smf.ols("RMSE_annual ~ mmd2_iceland_mean + log_n_meas",
                data=df_model).fit()

# Model B: Holdout distance only
res_b = smf.ols("RMSE_annual ~ mmd2_holdout_mean + log_n_meas",
                data=df_model).fit()

# Model C: Both (collinear but shows partial effects)
res_c = smf.ols(
    "RMSE_annual ~ mmd2_iceland_mean + mmd2_holdout_mean + log_n_meas",
    data=df_model).fit()

print("Model A (Iceland only)  — R²:", round(res_a.rsquared, 3),
      " | Iceland coef:", round(res_a.params["mmd2_iceland_mean"], 3), " p=",
      round(res_a.pvalues["mmd2_iceland_mean"], 3))

print("Model B (Holdout only)  — R²:", round(res_b.rsquared, 3),
      " | Holdout coef:", round(res_b.params["mmd2_holdout_mean"], 3), " p=",
      round(res_b.pvalues["mmd2_holdout_mean"], 3))

print("Model C (Both)          — R²:", round(res_c.rsquared, 3))